# Show, Attend and Tell — Colab training notebook (low-Drive version)

CS 4782, Spring 2026 — Iyer, Shaikh, George, Ying.

This notebook keeps the bulky data (COCO images, VGG features) on Colab **local** disk under `/content/` and only persists small artifacts (vocab, checkpoint, logs, attention PNGs) to Drive. Total Drive usage: ~200 MB.

**Runtime:** `Runtime > Change runtime type > A100 GPU > High-RAM`. If available, also enable `Runtime > Background execution`.

## 0. Setup

In [ ]:
# Mount Google Drive for persistent artifacts only (~200 MB total).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo (replace with your URL once pushed).
%cd /content
!git clone https://github.com/<user>/show-attend-tell-cs4782.git || (cd show-attend-tell-cs4782 && git pull)
%cd /content/show-attend-tell-cs4782
!pip install -q -r requirements.txt
# Extras needed for evaluation.
!apt-get install -y openjdk-17-jre-headless >/dev/null    # METEOR needs Java
!pip install -q pycocoevalcap
import nltk; nltk.download('punkt')

In [ ]:
import os

# -------- BULKY DATA: on Colab local disk (wiped at session end) --------
LOCAL_ROOT    = '/content/satell_local'
COCO_ROOT     = f'{LOCAL_ROOT}/coco2014'        # ~20 GB
FEATURES_PATH = f'{LOCAL_ROOT}/features/coco_vgg16.h5'  # ~15 GB

# -------- SMALL ARTIFACTS: on Drive (persistent) --------
DRIVE_ROOT     = '/content/drive/MyDrive/cs4782_show_attend_tell'
SPLITS_ROOT    = f'{DRIVE_ROOT}/karpathy_splits'
VOCAB_PATH     = f'{DRIVE_ROOT}/vocab.json'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints/run1'
RESULTS_DIR    = f'{DRIVE_ROOT}/results'

for d in (LOCAL_ROOT, f'{LOCAL_ROOT}/features', DRIVE_ROOT, SPLITS_ROOT,
          CHECKPOINT_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

print('local :', LOCAL_ROOT)
print('drive :', DRIVE_ROOT)
!df -h /content | tail -1
!df -h /content/drive 2>/dev/null | tail -1 || echo '(drive quota not shown)'

## 1. Data prep (only Karpathy splits + vocab persist; COCO is local)

Run these cells once per Colab session. If the session is fresh, all four cells run; if you're resuming an existing session, skip the ones whose outputs still exist.

In [ ]:
# Karpathy splits (small, persist to Drive).
if not os.path.exists(f'{SPLITS_ROOT}/dataset_coco.json'):
    !python -m code.data.prepare_karpathy --root {SPLITS_ROOT}
else:
    print('Karpathy splits already on Drive.')

In [ ]:
# Vocabulary (persist to Drive). Tiny file, runs in seconds.
if not os.path.exists(VOCAB_PATH):
    !python -m code.data.vocab \
        --splits {SPLITS_ROOT}/dataset_coco.json \
        --min-count 5 --max-size 10000 \
        --out {VOCAB_PATH}
else:
    print('Vocab already on Drive.')

In [ ]:
# COCO 2014 to LOCAL disk (~20 GB, ~20-30 min). This will NOT persist.
if not os.path.exists(f'{COCO_ROOT}/val2014'):
    !python -m code.data.download_coco --root {COCO_ROOT}
else:
    print('COCO already present locally.')
!df -h /content | tail -1

In [ ]:
# Precompute VGG features to LOCAL disk (~15 GB fp16, ~30-45 min).
if not os.path.exists(FEATURES_PATH):
    !python -m code.data.precompute_features \
        --coco-root {COCO_ROOT} \
        --splits {SPLITS_ROOT}/dataset_coco.json \
        --out {FEATURES_PATH} \
        --batch-size 128 --num-workers 4 --dtype fp16
else:
    print('Features already present locally.')
!ls -lh {FEATURES_PATH}

## 2. Smoke-test training loop (5–10 min)

One quick epoch to confirm the loss moves before committing to the 3–5 hour run.

In [ ]:
SMOKE_DIR = f'{DRIVE_ROOT}/checkpoints/smoke'
os.makedirs(SMOKE_DIR, exist_ok=True)
!python -m code.train \
    --features {FEATURES_PATH} \
    --splits {SPLITS_ROOT}/dataset_coco.json \
    --vocab {VOCAB_PATH} \
    --out-dir {SMOKE_DIR} \
    --batch-size 64 --epochs 1 --num-workers 2 \
    --log-every 50

## 3. Real training run (~3–5 hours, unattended)

Checkpoints land on Drive after every epoch — safe against disconnects.

In [ ]:
!python -m code.train \
    --features {FEATURES_PATH} \
    --splits {SPLITS_ROOT}/dataset_coco.json \
    --vocab {VOCAB_PATH} \
    --out-dir {CHECKPOINT_DIR} \
    --batch-size 128 --epochs 20 --lr 4e-4 \
    --lr-decay 0.8 --lr-decay-every 5 \
    --dropout 0.5 --doubly-stochastic-lambda 1.0 \
    --beam-size 3 --patience 4 --num-workers 4

## 4. Evaluation on Karpathy test split

In [ ]:
!python -m code.evaluate \
    --checkpoint {CHECKPOINT_DIR}/best.pt \
    --features {FEATURES_PATH} \
    --vocab {VOCAB_PATH} \
    --splits {SPLITS_ROOT}/dataset_coco.json \
    --split test --beam 3 \
    --out {RESULTS_DIR}/test_metrics.json

## 5. Generate every poster figure (one-shot)

Runs `code.generate_poster_figures`, which produces the following under `{RESULTS_DIR}/poster_figures/`:

- `training_curves.png` — train cross-entropy + val BLEU-4 / METEOR per epoch
- `results_table.png` + `results_table.md` — paper vs. ours comparison
- `sample_captions_good.png` — 6 highest-BLEU-4 test captions with image + reference
- `sample_captions_bad.png` — 6 lowest-BLEU-4 test captions (failure cases)
- `caption_length_hist.png` — reference vs. generated caption-length distribution
- `attention_examples/00..19.png` — 20 per-word attention visualizations

This is the only cell that needs to run after training + evaluation. Everything lands on Drive.

In [ ]:
POSTER_DIR = f'{RESULTS_DIR}/poster_figures'
os.makedirs(POSTER_DIR, exist_ok=True)

!python -m code.generate_poster_figures \
    --checkpoint   {CHECKPOINT_DIR}/best.pt \
    --features     {FEATURES_PATH} \
    --vocab        {VOCAB_PATH} \
    --splits       {SPLITS_ROOT}/dataset_coco.json \
    --coco-root    {COCO_ROOT} \
    --metrics-json {RESULTS_DIR}/test_metrics.json \
    --train-log    {CHECKPOINT_DIR}/train_log.jsonl \
    --out-dir      {POSTER_DIR} \
    --num-attn     20 \
    --beam-size    3

print('\n--- poster figures ---')
!ls -lh {POSTER_DIR}
!ls -lh {POSTER_DIR}/attention_examples | head -25

## 6. (Optional) Save a few raw val images to Drive

Keeps them around so you can make additional attention visualizations *after* the Colab session ends, without re-downloading COCO. Not required for the poster — the attention figures in Section 5 already persist to Drive.

In [ ]:
import shutil, glob, random
os.makedirs(f'{DRIVE_ROOT}/saved_val_images', exist_ok=True)
samples = sorted(glob.glob(f'{COCO_ROOT}/val2014/*.jpg'))[:2000]
random.seed(0)
picks = random.sample(samples, 8)
for img in picks:
    shutil.copy(img, f'{DRIVE_ROOT}/saved_val_images/')
print('saved', len(picks), 'images to Drive')